# 3. Preprocessing (Stage 2) - Create Assorted Dataset (MetaMathQA)

**Objective:** Load the trained VQ-VAE (from Stage 1) and use it to process the **MetaMathQA** training dataset.

This notebook creates the `metamath_assorted_standard.jsonl` file, which contains mixed sequences of `Prompt + Latent CoT + Solution`. This stage is crucial for fine-tuning Llama 3.2, as it integrates natural language with the discrete latent representations derived from the VQ-VAE. By utilizing the trained generator, we can characterize the data manifold through a structured latent space, effectively correcting distortions that occur in traditional linear Euclidean interpretations.

## 3.1 Environment Setup and Repository Cloning

To ensure reproducibility, this section automates the setup of the working environment:
1. **Google Drive Integration:** Mounts your personal Drive to store persistent data (checkpoints and processed datasets).
2. **Project Structure:** Automatically creates a `DLAI` folder in your Drive.
3. **Dependency Management:** Installs the `uv` package manager and resolves all requirements defined in `pyproject.toml`.
4. **Source Code:** Clones the `llama` branch from our GitHub repository to provide access to the `src` module and configuration files.

**Note for Evaluators:** Please authorize the Google Drive mount when prompted to allow the notebook to save and retrieve project files.

In [ ]:
import os, sys

# 1. Mount Google Drive
# Evaluators will need to accept the pop-up to connect their Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Setup directories on Drive
# Create the DLAI folder if it doesn't exist on their Drive
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/DLAI"
if not os.path.exists(DRIVE_PROJECT_PATH):
    os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
    print(f"Created project folder at: {DRIVE_PROJECT_PATH}")

# 3. UV Installation
# We use UV for much faster dependency management than standard pip
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = f"{os.path.expanduser('~')}/.cargo/bin:" + os.environ['PATH']

# 4. Clone the Repository (Branch: llama)
# If the local folder doesn't exist, clone the specific branch
%cd /content
if not os.path.exists("DLAI"):
    !git clone --branch llama https://github.com/irene-30/DLAI.git
else:
    print("Repo already exists, pulling latest changes...")
    !git -C DLAI pull

# 5. Synchronize pyproject.toml
# Copy the pyproject.toml from the cloned repo to the Drive folder (if necessary)
# or vice versa, to ensure that UV reads the correct dependencies.
!cp /content/DLAI/pyproject.toml {DRIVE_PROJECT_PATH}/pyproject.toml

# 6. Install dependencies via pyproject.toml
# This command reads the .toml file and installs everything necessary
%cd /content/DLAI
!uv pip install -e . --system

# 7. Add to the system path to allow imports from 'src'
sys.path.append("/content/DLAI")
%cd /content

print("✅ Setup completed successfully!")

Mounted at /content/drive
downloading uv 0.11.14 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
/content
Cloning into 'DLAI'...
remote: Enumerating objects: 640, done.
remote: Counting objects: 100% (194/194), done.
remote: Compressing objects: 100% (194/194), done.
remote: Total 640 (delta 136), reused 0 (delta 0), pack-reused 446 (from 2)
Receiving objects: 100% (640/640), 281.40 KiB | 11.25 MiB/s, done.
Resolving deltas: 100% (371/371), done.
/content/DLAI
Using Python 3.12.13 environment at: /usr
Resolved 88 packages in 758ms
Prepared 6 packages in 2.55s
Uninstalled 3 packages in 79ms
Installed 6 packages in 37ms
 + bitsandbytes==0.49.2
 - datasets==4.0.0
 + datasets==4.8.5
 + dlai-metamath==0.1.0 (from file:///content/DLAI)
 - pyarrow==18.1.0
 + pyarrow==24.0.0
 - torchao==0.10.0
 + torchao==0.17.0
 + trl==1.4.0
/content
✅ Setup completed successfully!


### Step 2: Configuration and Tokenizer Initialization

In this section, we initialize the Llama 3.2 tokenizer and define the storage paths. According to the geometric perspective, the latent space provides a distorted view of the input space that can be characterized by a Riemannian metric. The goal of this processing is to map MetaMathQA samples into this latent space to capture the underlying manifold structure.

In [ ]:
import torch
import json
from datasets import load_dataset
from src.utils import get_llm_tokenizer, MAX_SEQ_LEN, VQ_CODEBOOK_SIZE, create_assorted_dataset
from src.model.vae import VQVAEModel

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- UNIFORM PATH CONFIGURATION ---
# Path to the VQ-VAE model trained in the previous stage
PATH_VQVAE_MODEL = "/content/drive/My Drive/DLAI/experiments/vqvae_standard/vqvae_final.pth"

# Output folder for the processed assorted dataset
DRIVE_OUTPUT_FOLDER = "/content/drive/My Drive/DLAI/data/processed"
PATH_PROCESSED_DATA = os.path.join(DRIVE_OUTPUT_FOLDER, "metamath_assorted_standard.jsonl")

os.makedirs(DRIVE_OUTPUT_FOLDER, exist_ok=True)

# Initialize the LLM tokenizer
tokenizer = get_llm_tokenizer()
vocab_size = len(tokenizer)

Using device: cuda
Loading tokenizer: meta-llama/Llama-3.2-3B-Instruct


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

### Step 3: Load Trained VQ-VAE Model

We load the VQ-VAE model architecture and weights trained in Stage 1. This model acts as our generator $f(z)$. Mathematically, if the generator is sufficiently smooth, it defines a local Riemannian metric $M_z = J_z^T J_z$, where $J_z$ is the Jacobian. This metric governs how distances and interpolations are calculated within the latent representation.

In [ ]:
# 1. Instantiate the architecture (must match Stage 1 hyperparameters)
vq_model = VQVAEModel(
    vocab_size=vocab_size,
    d_model=256,
    num_embeddings=VQ_CODEBOOK_SIZE,
    max_seq_len=1024,
    commitment_cost=0.1
).to(device)

# 2. Load model weights with error handling
try:
    print(f"Load VQ-VAE from: {PATH_VQVAE_MODEL}")
    checkpoint = torch.load(PATH_VQVAE_MODEL, map_location=device)

    # Handle both full checkpoint dictionaries and raw state dicts
    if 'model_state_dict' in checkpoint:
        vq_model.load_state_dict(checkpoint['model_state_dict'])
    else:
        vq_model.load_state_dict(checkpoint)

    vq_model.eval()
    print("✅VQ-VAE model loaded successfully.")
except FileNotFoundError:
    print(f"❌ ERROR: Checkpoint not found at {PATH_VQVAE_MODEL}")

Caricamento VQ-VAE da: /content/drive/My Drive/DLAI/experiments/vqvae_standard/vqvae_final.pth
✅ Modello VQ-VAE caricato con successo.


### Step 4: Process MetaMathQA to Create Assorted Dataset

We process the MetaMathQA dataset to generate "assorted" samples. For each mathematical problem, the Chain-of-Thought (CoT) reasoning is encoded into discrete latent tokens using the VQ-VAE. This aligns with the idea that shortest paths on the surface spanned by the generator do not correspond to straight lines in the latent space; by providing these latent tokens, we help the LLM navigate the true data manifold.

In [ ]:
# Load MetaMathQA (using a subset for processing speed)
print("Loading MetaMathQA dataset...")
raw_dataset = load_dataset("meta-math/MetaMathQA")['train'].shuffle(seed=42).select(range(10000))

# Execute preprocessing to mix text and latent tokens
print("🚀 Creating Assorted Dataset (Standard VQ-VAE)...")
assorted_samples = create_assorted_dataset(
    vq_model=vq_model,
    llm_tokenizer=tokenizer,
    dataset=raw_dataset,
    device=device
)

print(f"\nGenerated {len(assorted_samples)} assorted samples.")

Caricamento dataset MetaMathQA...


README.md: 0.00B [00:00, ?B/s]

MetaMathQA-395K.json:   0%|          | 0.00/396M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/395000 [00:00<?, ? examples/s]

🚀 Creazione Assorted Dataset (Standard VQ-VAE)...
Creating 'Token Assorted' dataset (Standard VQ-VAE)...


100%|██████████| 10000/10000 [00:36<00:00, 271.47it/s]


Generati 9987 campioni assortiti.


### Step 5: Data Inspection and Verification

We verify the formatting of our processed samples. We specifically look for the presence of the special tokens `[boLatent]` and `<latent_X>`, which signify the transition into the Riemannian latent space where probability distributions and clustering are improved.

In [ ]:
# Verify the formatting of the first few samples
print("--- SAMPLE 1 ---")
print(assorted_samples[0]['text'][:500] + "...")

print("\n--- SAMPLE 2 ---")
print(assorted_samples[1]['text'][:500] + "...")

# Specifically look for a sample containing latent tokens
latent_sample = next((s['text'] for s in assorted_samples if "[boLatent]" in s['text']), "No latent sample found in batch.")
print("\n--- SAMPLE WITH LATENT TOKENS ---")
print(latent_sample[:1000] + "...")

--- SAMPLE 1 (May be text only if m=0) ---
Question: If Anna wants to create a smiley face shape using red and yellow tulips, she requires 8 red tulips for each eye and 18 red tulips for the smile. Additionally, she needs 9 times the number of tulips in the smile to create the yellow background of the face. What is the total number of tulips that Anna needs?
Answer: [boLatent] <latent_818> <latent_818> <latent_818> <latent_818> <latent_818> <latent_818> <latent_818> <latent_818> <latent_818> <latent_818> <latent_818> <latent_818> <latent...

--- SAMPLE 2 ---
Question: In the final game of the basketball season, four players scored points.  Chandra scored twice as many points as did Akiko.  Akiko scored 4 more points than did Michiko, and Michiko scored half as many points as did Bailey.  If Bailey scored 14 points, how many points in total did the team score in the final game of the season?
Answer: [boLatent] <latent_199> <latent_338> <latent_199> <latent_199> <latent_199> <latent_194>

### Step 6: Save Processed Data

The final assorted dataset is saved in `.jsonl` format. This file will be the primary input for fine-tuning Llama 3.2, enabling the model to perform reasoning that is grounded in both symbolic language and geometric latent representations.

In [ ]:
print(f"Saving processed data to: {PATH_PROCESSED_DATA}...")

with open(PATH_PROCESSED_DATA, 'w') as f:
    for item in assorted_samples:
        f.write(json.dumps(item) + '\n')

print(f"✅ Preprocessing complete. File saved at {PATH_PROCESSED_DATA}")

Salvataggio dati elaborati in: /content/drive/My Drive/DLAI/data/processed/metamath_assorted_standard.jsonl...
✅ Preprocessing completato. File salvato in /content/drive/My Drive/DLAI/data/processed/metamath_assorted_standard.jsonl
